# Toxicity in Indian esports YouTube comments

Runs the whole study: collect comments, score them, label a sample by hand, then measure and report.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.* Scoring is about 10× faster on a GPU.
2. Have a YouTube Data API key ready (free; see the README).

Steps 1–4 take about 15 minutes. Step 5 is the hand labelling, then step 6 produces the results.

In [ ]:
# 1. Get the code
!git clone -q https://github.com/harsh-github007/Identifying-Toxicity-Within-Esports-YouTube-Channels.git repo
%cd repo
!pip install -q -e ".[model]"

In [ ]:
# 2. API key: stored as a Colab secret named YOUTUBE_API_KEY (key icon in the left sidebar), or typed here
import os, getpass
try:
    from google.colab import userdata
    os.environ["YOUTUBE_API_KEY"] = userdata.get("YOUTUBE_API_KEY")
except Exception:
    os.environ["YOUTUBE_API_KEY"] = getpass.getpass("YouTube API key: ")

In [ ]:
# 3. Collect comments (edit config/channels.yaml first if a handle needs fixing)
!python -m toxicity collect

In [ ]:
# 4. Score every comment with the model and the word list, then draw the labelling sample
!python -m toxicity score
!python -m toxicity sample -n 400
from google.colab import files
files.download("data/annotation/to_label.csv")

## 5. Label the sample

Open `to_label.csv` in Google Sheets or Excel and fill the **toxic** column with `1` or `0` for every row,
following [docs/labelling-guide.md](https://github.com/harsh-github007/Identifying-Toxicity-Within-Esports-YouTube-Channels/blob/main/docs/labelling-guide.md).
The optional **category** column takes `insult`, `profanity`, `threat` or `hate`.

The sheet doesn't show which channel a comment came from or what the model thought, so the labels stay independent.
Save it as CSV, then upload it with the next cell.

In [ ]:
# Upload your labelled file
from google.colab import files
up = files.upload()
name = next(iter(up))
os.replace(name, "data/annotation/labels.csv")
print("Saved", name, "as data/annotation/labels.csv")

In [ ]:
# 6. Measure the methods, estimate toxicity per channel, draw the charts
!python -m toxicity evaluate
from IPython.display import Markdown, Image, display
display(Markdown(open("results/results.md").read().replace("figures/", "results/figures/")))
display(Image("results/figures/prevalence.png")); display(Image("results/figures/methods.png"))

In [ ]:
# 7. Download the results to add to the repository (results/ and your labels)
!zip -qr results_to_commit.zip results data/annotation/labels.csv data/annotation/sample_key.csv config
files.download("results_to_commit.zip")